In [1]:
#Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ML_Modelevaluation import SoilModel, HSdata_process

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor

In [2]:
######################## Define the text size of each plot globally ###########
SMALL_SIZE = 10
BIGGER_SIZE = 10

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=SMALL_SIZE)     # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

plt.rcParams["font.family"] = "Arial"
######################## Define the text size of each plot globally ###########

In [3]:
# =============================================================================
# Input structure HS - needs user interaction
# =============================================================================

input_column = ['E50ref', 'Eoedref', 'Eurref', 
                'phi','cref', 'psi', 
                'm', 'nu', 
                'Rf', 'K0NC', 'CellPressure']

triaxial_column = ['q', 'eps_y', 
                   'eps_v', 'p']

odometer_column = ['sig_y', 'eps_y_oed']

cell_pressure = [200]

In [4]:
# =============================================================================
# Modelevaluation HS
# ====================================================================

#input data for soil model
HS = HSdata_process(modelname='HardeningSoil_Database_20240715', 
                    CellPressure='CellPressure',
                    cellpressure_value=cell_pressure,
                    input_column= input_column,
                    triaxial_column = triaxial_column,
                    odoemeter_column= odometer_column)

input_parameters, output_triaxial, output_odometer, input_parameters_nan = HS.data_processing()

In [5]:
input_parameters = input_parameters.astype(float)

In [6]:
# Reference pressure - 100 kPa  
input_parameters.insert(loc=11, column='pref', value=float(100.0))
input_column.append('pref')

In [7]:
input_parameters

,E50ref,Eoedref,Eurref,phi,cref,psi,m,nu,Rf,K0NC,CellPressure,pref
0,47015.078131,37612.0,94030.2,36.942514,0.0,6.942514,0.457175,0.258691,0.750753,0.398987,200.0,100.0
2,69152.751019,55322.2,138305.6,30.196669,0.0,0.196669,0.667083,0.225450,0.951141,0.497030,200.0,100.0
3,28709.171082,22967.3,57418.4,34.301510,0.0,4.301510,0.445161,0.124509,0.889554,0.436452,200.0,100.0
4,29401.355872,23521.0,58802.8,33.564483,0.0,3.564483,0.523623,0.295160,0.867171,0.447125,200.0,100.0
5,50465.270589,40372.2,100930.6,35.461038,0.0,5.461038,0.400504,0.167644,0.859915,0.419851,200.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...
40494,15646.914401,18776.2,62587.7,31.524586,0.0,0.000000,0.645787,0.183601,0.729768,0.477136,200.0,100.0
40495,51792.300857,62150.7,207169.3,28.458506,0.0,0.000000,0.603964,0.296719,0.783701,0.523478,200.0,100.0
40497,22878.896370,27454.6,91515.6,35.887609,0.0,0.000000,0.320329,0.170195,0.749667,0.413803,200.0,100.0
40498,27185.502230,32622.6,108742.1,33.554801,0.0,0.000000,0.471491,0.265357,0.807118,0.447266,200.0,100.0


In [8]:
def x_array(x_matrix):
    X_return = []
    for s in x_matrix:
        # Remove '[' and ']' characters, split by '\n' to get individual lines
        s = str(s)
        lines = s.strip('[]').split(',')
        values = []
        for line in lines:
            # Split each line by spaces, convert values to floats, and extend the list
            values.extend([float(val) for val in line.split()])   
        X_return.append(np.array(values))
    X_return = np.array(X_return)
    return X_return

#triaxial test
q = x_array(output_triaxial.iloc[:,0])
eps_y = x_array(output_triaxial.iloc[:,1])
eps_vol = x_array(output_triaxial.iloc[:,2])
p = x_array(output_triaxial.iloc[:,3])

#oedometer test
sig_1 = x_array(output_odometer.iloc[:,0])
eps_y_oed = x_array(output_odometer.iloc[:,1])

In [9]:
soil_model = SoilModel()

df_sig = pd.DataFrame()
df_eps = pd.DataFrame()

for sig, eps in zip(sig_1, eps_y_oed):
    min_index = np.argmin(sig)
    degree = 15

    # Loading
    eps1_section_1 = eps[:min_index + 1]
    sig1_section_1 = sig[:min_index + 1]

    # Unloading
    eps1_section_2 = eps[min_index:]
    sig1_section_2 = sig[min_index:]

    # Filter for sigma not less than -50
    mask = sig1_section_2 <= -50
    cutoff_indices = np.where(~mask)[0]  # Indices that do not meet the condition
    
    if len(cutoff_indices) > 0:
        #print("Indices being cut off for sig1_section_2:", cutoff_indices)
        eps1_section_2 = eps1_section_2[:cutoff_indices[0]]
        sig1_section_2 = sig1_section_2[:cutoff_indices[0]]
    
    eps1_section_1_syn = np.linspace(0, min(eps1_section_1), num=376)
    eps1_section_2_syn = np.linspace(min(eps1_section_1), max(eps1_section_2), num=376)

    sig1_pred_section_1 = soil_model.interpolation_scikit(x_true=eps1_section_1, y_true=sig1_section_1, x_check=eps1_section_1_syn, degree=degree)
    sig1_pred_section_2 = soil_model.interpolation_scikit(x_true=eps1_section_2, y_true=sig1_section_2, x_check=eps1_section_2_syn, degree=degree)

    # Convert to DataFrame and transpose
    sig1_pred_section_1_df = pd.DataFrame(sig1_pred_section_1).T
    sig1_pred_section_2_df = pd.DataFrame(sig1_pred_section_2).T
    eps1_section_1_syn_df = pd.DataFrame(eps1_section_1_syn).T
    eps1_section_2_syn_df = pd.DataFrame(eps1_section_2_syn).T

    # Concatenate horizontally
    sig_combined_df = pd.concat([sig1_pred_section_1_df, sig1_pred_section_2_df], axis=1)
    eps_combined_df = pd.concat([eps1_section_1_syn_df, eps1_section_2_syn_df], axis=1)

    df_sig = pd.concat([df_sig, sig_combined_df], ignore_index=True)
    df_eps = pd.concat([df_eps, eps_combined_df], ignore_index=True)

In [10]:
df_q = pd.DataFrame()
df_p = pd.DataFrame()

i = 0
for q_a, p_a in zip(q, p):
    max_q, min_q = max(q_a), min(q_a)
    max_p, min_p = max(p_a), min(p_a)
    
    q_export = np.linspace(min_q, max_q, num=251)
    p_export = np.linspace(max_p, min_p, num=251)

    # Convert to DataFrame and transpose
    q_export = pd.DataFrame(q_export).T
    p_export = pd.DataFrame(p_export).T

    # Concatenate horizontally
    df_q = pd.concat([df_q, q_export], axis=0)
    df_p = pd.concat([df_p, p_export], axis=0)

In [11]:
# Check for NaN values in df_sig
nan_in_df_sig = df_sig.isna().any().any()
print(f"NaN values in df_sig: {nan_in_df_sig}")

# Check for NaN values in df_eps
nan_in_df_eps = df_eps.isna().any().any()
print(f"NaN values in df_eps: {nan_in_df_eps}")

NaN values in df_sig: False
NaN values in df_eps: False


In [12]:
X_triax = np.hstack((q, eps_y, eps_vol, eps_y))
X_triax = pd.DataFrame(X_triax)

# Ensure that all DataFrames have unique indices
df_q = df_q.reset_index(drop=True)
df_p = df_p.reset_index(drop=True)
X_triax = X_triax.reset_index(drop=True)

# Concatenate DataFrames
X_triax = pd.concat([X_triax, df_q, df_p], ignore_index=False, sort=False, axis=1)
X_oedo = pd.concat([df_sig, df_eps], ignore_index = True, sort = False, axis = 1)

In [13]:
X = pd.concat([X_triax, X_oedo], ignore_index = True, sort = False, axis = 1)
y = input_parameters

In [14]:
X

,0,1,2,3,4,5,6,7,8,9,...,2996,2997,2998,2999,3000,3001,3002,3003,3004,3005
0,0.0,93.26830,164.18549,221.95807,270.59355,312.32252,348.56347,380.32557,408.36215,433.25897,...,-0.006814,-0.006806,-0.006797,-0.006788,-0.006780,-0.006771,-0.006763,-0.006754,-0.006745,-0.006737
1,0.0,135.19789,205.80227,251.43260,282.86647,305.47981,322.32796,335.26250,345.45055,353.65789,...,-0.005619,-0.005615,-0.005610,-0.005605,-0.005600,-0.005595,-0.005590,-0.005586,-0.005581,-0.005576
2,0.0,66.51164,116.87364,155.90200,188.72435,217.06242,241.92024,263.95612,283.63835,301.31991,...,-0.011370,-0.011356,-0.011342,-0.011328,-0.011314,-0.011300,-0.011287,-0.011273,-0.011259,-0.011245
3,0.0,67.57117,118.42081,159.57555,194.11551,223.72066,249.44197,272.00787,291.94921,309.67680,...,-0.011599,-0.011586,-0.011574,-0.011561,-0.011549,-0.011536,-0.011523,-0.011511,-0.011498,-0.011486
4,0.0,101.15762,170.09241,223.49692,266.91478,303.08653,333.67900,359.83510,382.39166,401.98726,...,-0.006358,-0.006350,-0.006342,-0.006334,-0.006326,-0.006318,-0.006311,-0.006303,-0.006295,-0.006287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34006,0.0,37.70723,70.13712,98.58523,123.91356,146.71424,167.41247,186.32573,203.69922,219.72778,...,-0.017054,-0.017043,-0.017032,-0.017020,-0.017009,-0.016998,-0.016987,-0.016975,-0.016964,-0.016953
34007,0.0,100.08269,164.17317,210.05852,244.59925,271.44335,292.82114,310.17001,324.49597,336.50006,...,-0.005174,-0.005171,-0.005168,-0.005165,-0.005162,-0.005159,-0.005156,-0.005153,-0.005150,-0.005147
34008,0.0,42.63763,80.74051,115.05960,146.16584,174.50402,200.43395,224.25034,246.19913,266.48819,...,-0.010228,-0.010219,-0.010210,-0.010201,-0.010193,-0.010184,-0.010175,-0.010166,-0.010157,-0.010148
34009,0.0,56.96362,104.01111,143.81023,178.02944,207.80272,233.94961,257.09040,277.70554,296.17612,...,-0.008992,-0.008986,-0.008979,-0.008972,-0.008966,-0.008959,-0.008952,-0.008946,-0.008939,-0.008932


In [15]:
y

,E50ref,Eoedref,Eurref,phi,cref,psi,m,nu,Rf,K0NC,CellPressure,pref
0,47015.078131,37612.0,94030.2,36.942514,0.0,6.942514,0.457175,0.258691,0.750753,0.398987,200.0,100.0
2,69152.751019,55322.2,138305.6,30.196669,0.0,0.196669,0.667083,0.225450,0.951141,0.497030,200.0,100.0
3,28709.171082,22967.3,57418.4,34.301510,0.0,4.301510,0.445161,0.124509,0.889554,0.436452,200.0,100.0
4,29401.355872,23521.0,58802.8,33.564483,0.0,3.564483,0.523623,0.295160,0.867171,0.447125,200.0,100.0
5,50465.270589,40372.2,100930.6,35.461038,0.0,5.461038,0.400504,0.167644,0.859915,0.419851,200.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...
40494,15646.914401,18776.2,62587.7,31.524586,0.0,0.000000,0.645787,0.183601,0.729768,0.477136,200.0,100.0
40495,51792.300857,62150.7,207169.3,28.458506,0.0,0.000000,0.603964,0.296719,0.783701,0.523478,200.0,100.0
40497,22878.896370,27454.6,91515.6,35.887609,0.0,0.000000,0.320329,0.170195,0.749667,0.413803,200.0,100.0
40498,27185.502230,32622.6,108742.1,33.554801,0.0,0.000000,0.471491,0.265357,0.807118,0.447266,200.0,100.0


In [16]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_eval, y_test, y_eval = train_test_split(X_temp, y_temp, test_size=0.33, random_state=42)

In [17]:
results = pd.DataFrame(columns=['Parameters','Decision Tree', 'Random Forest', 'HistGradient Boosting Regressor', 'XGBRegressor' ])
results['Parameters'] = y.columns

In [18]:
regr1 = DecisionTreeRegressor(random_state=42)
regr1.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

In [19]:
y_pred = regr1.predict(X_test)
y_pred

score_all = []
for i in range(y_test.shape[1]):
    y_true_col = y_test.values[:, i]
    y_pred_col = y_pred[:, i]
    
    # Renaming the variable to avoid conflict with the function name
    r2_score,mse = HS.eval_error(y_true = y_true_col, y_pred = y_pred_col)
    score_all.append(r2_score)

results['Decision Tree'] = score_all

c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: invalid value encountered in divide
  output_scores = 1 - (numerator / denominator)
c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: invalid value encountered in divide
  output_scores = 1 - (numerator / denominator)
c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: invalid value encountered in divide
  output_scores = 1 - (numerator / denominator)


In [20]:
regr2 = RandomForestRegressor(random_state=42)
regr2.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [21]:
y_pred = regr2.predict(X_test)

score_all = []
for i in range(y_pred.shape[1]):
    y_true_col = y_test.values[:, i]
    y_pred_col = y_pred[:, i]

    r2_score,mse = HS.eval_error(y_true_col, y_pred_col)
    score_all.append(r2_score)

results['Random Forest'] = score_all

c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: invalid value encountered in divide
  output_scores = 1 - (numerator / denominator)
c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: invalid value encountered in divide
  output_scores = 1 - (numerator / denominator)
c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: invalid value encountered in divide
  output_scores = 1 - (numerator / denominator)


In [22]:
regr3 = HistGradientBoostingRegressor(random_state=42)
multi_target_regr = MultiOutputRegressor(regr3)
multi_target_regr.fit(X_train, y_train)

MultiOutputRegressor(estimator=HistGradientBoostingRegressor(random_state=42))

In [23]:
y_pred = multi_target_regr.predict(X_test)
score_all = []
for i in range(y_pred.shape[1]):
    y_true_col = y_test.values[:, i]
    y_pred_col = y_pred[:, i]

    r2_score,mse = HS.eval_error(y_true_col, y_pred_col)
    score_all.append(r2_score)

results['HistGradient Boosting Regressor'] = score_all

c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: invalid value encountered in divide
  output_scores = 1 - (numerator / denominator)
c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: invalid value encountered in divide
  output_scores = 1 - (numerator / denominator)
c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: invalid value encountered in divide
  output_scores = 1 - (numerator / denominator)


In [24]:
regr4 = XGBRegressor(random_state=42)
regr4.fit(X_train, y_train, eval_set=[(X_train, y_train),(X_eval,y_eval)], eval_metric='rmse', verbose=False)

c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\xgboost\sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [25]:
y_pred = regr4.predict(X_test)
score_all = []
for i in range(y_pred.shape[1]):
    y_true_col = y_test.values[:, i]
    y_pred_col = y_pred[:, i]

    r2_score,mse = HS.eval_error(y_true_col, y_pred_col)
    score_all.append(r2_score)

results['XGBRegressor'] = score_all

c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: divide by zero encountered in divide
  output_scores = 1 - (numerator / denominator)
c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: divide by zero encountered in divide
  output_scores = 1 - (numerator / denominator)
c:\Users\hari1996\Documents\GitHub\MLpFEM\MLpFEM\Lib\site-packages\sklearn\metrics\_regression.py:624: RuntimeWarning: divide by zero encountered in divide
  output_scores = 1 - (numerator / denominator)


In [26]:
results = results.round(4)
results.to_csv('results.csv', index=False)
results

,Parameters,Decision Tree,Random Forest,HistGradient Boosting Regressor,XGBRegressor
0,E50ref,0.9929,0.9980,0.9986,0.9989
1,Eoedref,0.9983,0.9997,0.9999,0.9999
2,Eurref,0.9801,0.9918,0.9825,0.9898
3,phi,0.8111,0.9268,1.0000,1.0000
4,cref,NaN,NaN,NaN,-inf
5,psi,-1.1363,-0.2664,0.9999,0.9997
6,m,0.9049,0.9708,0.9999,0.9999
7,nu,0.6364,0.8047,0.9936,0.9942
8,Rf,0.5443,0.7289,0.9584,0.9639
9,K0NC,0.8113,0.9269,1.0000,1.0000
